In [1]:
import pandas as pd
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", None)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

# 01 - Data Overview
Load the 5 Olist tables used in the project, explore each one, and merge them into a the final dataset

In [3]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")

# 1. Explore Tables


In [4]:
def explore(df, name):
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head())
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })
    display(summary)

## 1.1 Orders

In [5]:
explore(orders, "orders")
print("order_id unique:", orders["order_id"].is_unique)

orders: 99,441 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,dtype,missing,missing_%,unique
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,str,0,0.00,98875
order_approved_at,str,160,0.16,90733
order_delivered_carrier_date,str,1783,1.79,81018
order_delivered_customer_date,str,2965,2.98,95664
order_estimated_delivery_date,str,0,0.00,459


order_id unique: True


**Orders** 
| Column | Type | Description |
|---|---|---|
| `order_id` | ID | Unique order identifier (primary key) |
| `customer_id` | ID | Customer key for this order (one per order, links to customers) |
| `order_status` | Categorical | Order status (delivered, shipped, canceled, ...) |
| `order_purchase_timestamp` | Date | When the order was placed |
| `order_approved_at` | Date | When the payment was approved |
| `order_delivered_carrier_date` | Date | When the order was handed to the carrier |
| `order_delivered_customer_date` | Date | When the customer received the order |
| `order_estimated_delivery_date` | Date | Estimated delivery date shown at purchase |

## 1.2 Order Items

In [6]:
explore(order_items, "order_items")
print("Max items in one order:", order_items["order_item_id"].max())

order_items: 112,650 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


,dtype,missing,missing_%,unique
order_id,str,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,str,0,0.0,32951
seller_id,str,0,0.0,3095
shipping_limit_date,str,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999


Max items in one order: 21


**Order Items** (one row per product within an order )
| Column | Type | Description |
|---|---|---|
| `order_id` | ID | Order identifier (links to orders) |
| `order_item_id` | ID | Item number within the order (1, 2, 3, ...) |
| `product_id` | ID | Product identifier (links to products) |
| `seller_id` | ID | Seller identifier |
| `shipping_limit_date` | Date | Deadline for the seller to ship the item |
| `price` | Numerical | Item price (BRL) |
| `freight_value` | Numerical | Shipping cost of the item (BRL) |

## 1.3 Products

In [7]:
explore(products, "products")
print("product_id unique:", products["product_id"].is_unique)

products: 32,951 rows x 9 columns


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


,dtype,missing,missing_%,unique
product_id,str,0,0.00,32951
product_category_name,str,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95


product_id unique: True


**Products** (one row per product)

| Column | Type | Description |
|---|---|---|
| `product_id` | ID | Unique product identifier (primary key) |
| `product_category_name` | Categorical | Product category  |
| `product_name_lenght` | Numerical | Number of characters in the product name |
| `product_description_lenght` | Numerical | Number of characters in the description |
| `product_photos_qty` | Numerical | Number of product photos |
| `product_weight_g` | Numerical | Weight in grams |
| `product_length_cm` | Numerical | Length in cm |
| `product_height_cm` | Numerical | Height in cm |
| `product_width_cm` | Numerical | Width in cm |


## 1.4 Customers

In [8]:
explore(customers, "customers")
print("customer_id unique:", customers["customer_id"].is_unique)
print("Unique real customers:", customers["customer_unique_id"].nunique())

customers: 99,441 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


,dtype,missing,missing_%,unique
customer_id,str,0,0.0,99441
customer_unique_id,str,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,str,0,0.0,4119
customer_state,str,0,0.0,27


customer_id unique: True
Unique real customers: 96096


**Customers** (one row per order's customer key)

| Column | Type | Description |
|---|---|---|
| `customer_id` | ID | Customer key generated per order (links to orders) |
| `customer_unique_id` | ID | Real customer identifier (same person across orders) |
| `customer_zip_code_prefix` | Numerical | First 5 digits of the zip code |
| `customer_city` | Categorical | Customer city |
| `customer_state` | Categorical | Customer state (2-letter code) |


## 1.5 Payments

In [9]:
explore(payments, "payments")
print("Orders with more than one payment:", (payments["order_id"].value_counts() > 1).sum())

payments: 103,886 rows x 5 columns


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


,dtype,missing,missing_%,unique
order_id,str,0,0.0,99440
payment_sequential,int64,0,0.0,29
payment_type,str,0,0.0,5
payment_installments,int64,0,0.0,24
payment_value,float64,0,0.0,29077


Orders with more than one payment: 2961


**Payments** (one row per payment method used in an order)

| Column | Type | Description |
|---|---|---|
| `order_id` | ID | Order identifier (links to orders) |
| `payment_sequential` | Numerical | Payment number when an order uses several methods |
| `payment_type` | Categorical | Payment method (credit_card, boleto, voucher, debit_card) |
| `payment_installments` | Numerical | Number of installments |
| `payment_value` | Numerical | Amount paid with this method (BRL) |

*Note: an order can have several payments, so payments are grouped by order before the merge (section 2.1).*

# 2. Merge Tables
Inner joins following the ERD. The final table has one row per product sold.



## 2.1 Group Payments by Order
An order can be paid with several methods. Joining `payments` as it is would repeat the item rows once per payment, so first we keep one row per order with the total paid.

In [10]:
payments_order = (
    payments.sort_values("payment_sequential")
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", "first"),
        n_payments=("payment_sequential", "count"),
    )
    .reset_index()
)
print(f"payments: {len(payments):,} rows -> payments_order: {len(payments_order):,} rows")

payments: 103,886 rows -> payments_order: 99,440 rows


## 2.2 Joins

In [11]:
df = orders.merge(order_items, on="order_id", how="inner")
print("+ order_items:", df.shape)

df = df.merge(products, on="product_id", how="inner")
print("+ products:   ", df.shape)

df = df.merge(customers, on="customer_id", how="inner")
print("+ customers:  ", df.shape)

df = df.merge(payments_order, on="order_id", how="inner")
print("+ payments:   ", df.shape)

+ order_items: (112650, 14)
+ products:    (112650, 22)
+ customers:   (112650, 26)
+ payments:    (112647, 30)


In [12]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments,payment_type,n_payments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,1,credit_card,3
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1,boleto,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,3,credit_card,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1,credit_card,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1,credit_card,1


**Merged table** (one row per product sold)

| Group | Columns |
|---|---|
| Identifiers (6) | `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `customer_unique_id` |
| Dates (6) | `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`, `shipping_limit_date` |
| Numerical (13) | `price`, `freight_value`, `product_name_lenght`, `product_description_lenght`, `product_photos_qty`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`, `customer_zip_code_prefix`, `payment_value`, `payment_installments`, `n_payments` |
| Categorical (5) | `order_status`, `product_category_name`, `customer_city`, `customer_state`, `payment_type` |

*Note: `payment_value` is the total of the order, so it repeats on every product of the same order. Group by `order_id` before summing it.*

# 3. Save
Save the merged table for the cleaning notebook.

In [13]:
df.to_csv(PROCESSED / "orders_merged.csv", index=False)